# Beyond Compliance: Structured Knowledge from Reports

Transform unstructured sustainability and regulatory disclosures into queryable graphs.

### The Challenge

Across industries, reporting is moving toward more standardised, machine-readable disclosures. Yet most critical information remains unstructured making consistency, comparability, and verification difficult.  
In sustainability reporting, frameworks such as CSRD are introducing digital formats and XBRL-based tagging for structured disclosures.  
However, even when datapoints are tagged, organisations still face recurring issues: inconsistent units or scales, unclear calculation methods, and fragmented narrative context across tables, notes and supporting materials.  

### The Solution: Ontology-Guided Extraction

Perseus Text-to-Graph uses an LLM constrained by a domain ontology to extract and structure quantitative and narrative disclosures into a unified graph model.

**Why the ontology matters:**
- **Semantic normalisation**: All units, scopes, and periods standardized (e.g., all emissions in tCO2e, not mixed kg/Mt)
- **Structured relationships**: Explicit links between metrics and their context (Program → Pillar → Target)  
- **Comprehensive coverage**: Works on both structured reports and unstructured materials (press releases, policies, etc.)

Without an ontology, LLMs produce unstructured JSON. With it, you get queryable, CSRD-compliant knowledge graphs.

### This Notebook

We process **2 contrasting climate reports** through Perseus Text-to-Graph to extract **ESRS E1 climate data** (the European climate change reporting standard under CSRD - Corporate Sustainability Reporting Directive):
- **EcoSteel Industries** (heavy industry, 3.8M tCO2e) - annual sustainability report with GHG inventory
- **TechGreen Solutions** (digital services, 58k tCO2e) - press release with Net-Zero commitment

The notebook walks through:
1. **Setup**: Load and upload a CSRD-aligned ontology (400+ classes covering E, S, G pillars)
2. **Extraction**: Convert both documents into RDF knowledge graphs (.ttl files)
3. **Exploration**: Inspect extracted entities with SPARQL, see concrete transformation examples (source text → ontology-guided, structured entities)
4. **Verification**: Run automated ESRS E1 compliance checks via Cypher queries on Neo4j

**Note**: SPARQL and Cypher queries are defined in `queries.py` for reusability. Key examples are shown inline for educational purposes.

---

#### Prerequisites

1. **Local Neo4j** (optional, for Section 6 verification queries)

```bash
# setup your username/password in .env
docker run -d --name neo4j-finance-compliance-demo -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password neo4j:latest 
```

2. **`.env` file** with `PERSEUS_API_KEY=your_key` (Create your API key [here](https://app.perseus.lettria.net/app/api-keys))

In [ ]:
# Optional: Install dependencies if not already installed
# rdflib and neo4j are automatically installed as perseus-client dependencies
# !pip install perseus-client python-dotenv

## 1. Inspect Lettria' CSRD Ontology

Lettria's `ontology_CSRD_V2.ttl` ontology models the sustainability reporting domain.  
It was automatically generated using our `Ontology Toolkit` solution. Find more ontologies on [Lettria resources](https://www.lettria.com/lettria-lab?categorie=Ontology+Management).  

- **400 classes** covering all 3 ESG pillars (Environment, Social, Governance)
- **160+ relationships** to link entities together
- **OWL/Turtle format** (.ttl) - compatible with [Protégé](https://protege.stanford.edu/).

#### Examples of ESRS E1 classes (Climate)

These classes will guide the extraction of climate data from unstructured reports:

| Ontology Class | Maps to ESRS | Purpose |
|----------------|--------------|---------|
| `GHGEmissionsMetric` | E1-6 | GHG emissions by scope (1, 2, 3) |
| `CarbonIntensityMetric` | E1-5 | Carbon intensity per unit |
| `Target`, `Commitment` | E1-4 | Net-Zero and reduction targets |
| `ClimateRisk` | E1-1 | Physical and transition risks |

#### Examples of Relationships
`hasMetric`, `hasTarget`, `hasCommitment`, `hasRisk`

Full ontology is available in `assets/ontology_CSRD_V2.ttl`

In [4]:
# Load and inspect the ontology, available in assets/
from pathlib import Path
from rdflib import Graph, RDF, OWL

ontology_path = Path("assets/ontology_CSRD_V2.ttl")

g = Graph()
g.parse(ontology_path, format="turtle")

# Count classes and properties
classes = set(g.subjects(RDF.type, OWL.Class))
properties = set(g.subjects(RDF.type, OWL.ObjectProperty)) | set(
    g.subjects(RDF.type, OWL.DatatypeProperty)
)

print(f"CSRD_V2 Ontology loaded: {len(classes)} classes, {len(properties)} properties")

CSRD_V2 Ontology loaded: 412 classes, 304 properties


## 2. Upload the Ontology to Perseus Platform

Optional : we'll upload the ontology to the Perseus Platform.  
**Note**:  Later, when performing text-to-graph from your files with `build_graph()` you can also upload the ontology (or decide not to) if you pass the `ontology_path` argument.   
Here we upload it explicitly to get its ID and link to the Perseus interface. The SDK automatically deduplicates by content (SHA256).

In [ ]:
from perseus_client.client import PerseusClient


# Upload ontology
with PerseusClient() as client:
    print("Uploading ontology...")

    ontology = client.ontology.upload_ontology(str(ontology_path))
    print(f"Ontology ID: {ontology.id} - Status: {ontology.status.value}")

    client.ontology.wait_for_ontology_upload(ontology.id)

    ontology_id = ontology.id
    print(f"\nView in Perseus: https://app.perseus.lettria.net/app/ontologies/")

RuntimeError: Cannot run the event loop while another loop is running

## 3. Guided text-to-graph on First Document: EcoSteel 2024 Climate Report

**🏭 EcoSteel Industries** - Heavy industry (steel manufacturing, 3.8M tCO2e annual emissions) investing €850M in its "GreenSteel 2030" decarbonization program.

**Document:** `annual_report.md` - Annual sustainability report with full GHG inventory, carbon intensity metrics, and strategic initiatives.

*Sample extracts:*
> "**Scope 1**: 1,200,000 tCO2e (direct combustion, blast furnaces)"  
> "**GreenSteel 2030 Program** - Pillar 1: Process Electrification"  
> "**Physical risk: Water stress** - Mediterranean sites vulnerable to drought"

**Outputs after processing:**
- `.ttl` file: RDF knowledge graph for semantic queries (SPARQL)
- `.cql` file: Cypher commands to load into Neo4j

*Processing typically takes 30-60 seconds. Track progress at https://app.perseus.lettria.net/app/jobs*

**Neo4j setup :** The jobs below use `save_to_neo4j=True` to automatically load extracted entities into your local Neo4j database for graph queries.  
This is optional. Files (.ttl, .cql) are generated regardless.

If you want to use Neo4j, ensure it's running locally :
```bash
docker ps | grep neo4j  # Check if running
# If not running:
docker start neo4j  # Or: docker run -d --name neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j:latest
```

If Neo4j isn't running, you'll see an authentication error (job still succeeds, files are generated).

In [2]:
# Process EcoSteel climate report (text-to-graph)
async with PerseusClient() as client:
    doc1_path = Path("assets/annual_report.md")

    print("Processing: Climate Report 2024 - EcoSteel Industries")
    print(f"File: {doc1_path}")

    job = await client.build_graph_async(
        file_path=str(doc1_path),
        ontology_path=str(ontology_path),
        output_path="output/ecosteel_climate_graph",
        save_to_neo4j=True,
    )

    print(f"Job ID: {job.id} - Status: {job.status.value}")
    print(f"View in Perseus: https://app.perseus.lettria.net/app/jobs/{job.id}")

    job1_id = job.id

NameError: name 'PerseusClient' is not defined

## 4. Guided text-to-graph on Second Document: TechGreen 2025 Climate Commitment

**☁️ TechGreen Solutions** - Digital services (cloud & data centers, 58k tCO2e) committing to Net-Zero 2040 with a €75M "Digital4Climate" program.

**Document:** `press_release.md` - Forward-looking press release focused on commitments, targets, and strategic programs (less quantitative than EcoSteel's annual report).

*Sample extracts:*
> "**Net-Zero 2040 Commitment** - Application submitted to SBTi"  
> "**Digital4Climate Program** - Pillar 1: Green Data Centers - 100% renewable electricity from 2026"  
> "**Transition risk: Technological obsolescence** - High PUE (Data centers' Power Usage Effectiveness) equipment"

In [1]:
# Process TechGreen climate commitment
async with PerseusClient() as client:
    doc2_path = Path("assets/press_release.md")

    print("Processing: Climate Commitment 2025 - TechGreen Solutions")
    print(f"File: {doc2_path}")

    job = await client.build_graph_async(
        file_path=str(doc2_path),
        ontology_path=str(ontology_path),
        output_path="output/techgreen_climate_graph",
        save_to_neo4j=True,
    )

    print(f"Job ID: {job.id} - Status: {job.status.value}")
    print(f"View in Perseus: https://app.perseus.lettria.net/app/jobs/{job.id}")

    job2_id = job.id

print("\n" + "=" * 60)
print("✅ Documents processed")
print(f"EcoSteel: {job1_id}")
print(f"TechGreen: {job2_id}")

NameError: name 'PerseusClient' is not defined

## Visualize in the Perseus Interface

The outputs are accessible both locally (`.ttl` and `.cql` files) and in the **[Perseus web interface](https://app.perseus.lettria.net/app/jobs)** for interactive exploration:

- **Nodes tab**: Browse all extracted entities (metrics, programs, risks, targets)
- **Graph tab**: Visualize relationships interactively
- **Turtle and Cypher tab**: Download Turtle (.ttl) and Cypher (.cql) files



**When to use the Perseus UI:**
- Debug extraction results before writing custom queries
- Share findings with non-technical stakeholders (visual graph exploration)
- Explore the ontology structure and extracted relationships

Go to **https://app.perseus.lettria.net/app/jobs** to view your processing jobs.  
You can also upload your own ontology and process documents directly through the interface.

**Example: Perseus job interface showing extracted entities**

![Perseus interface - Jobs view with extracted climate entities](assets/perseus_viz_jobs.png)

*The Perseus interface provides interactive exploration of extracted entities: browse nodes, visualize relationships in graph format, and download Turtle (.ttl) or Cypher (.cql) files for further analysis.*

## 5. Inspect Extracted RDF

Before running compliance checks, let's verify what Perseus extracted from **EcoSteel's climate report**. We'll use SPARQL queries directly on the `.ttl` file to explore key climate reporting entities.

This section focuses on the most important entity types for ESRS E1 compliance:
- **Quantitative metrics**: GHG emissions by scope, carbon intensity
- **Strategic elements**: Programs and pillars for decarbonization
- **Risk assessments**: Physical and transition climate risks
- **Commitments**: Net-Zero targets and reduction goals

The output shows how unstructured text was transformed into structured, queryable RDF triples—ready for automated compliance verification.

**Note**: We're using SPARQL to query the RDF file directly (no Neo4j required for this exploration).

### SPARQL Query Examples

The queries used in this notebook are defined in `queries.py`. Here an example exploring RDF data:

**SPARQL - Extract GHG emissions metrics:**
```sparql
SELECT ?label ?value ?unit ?year
WHERE {
    ?metric a ?type .
    FILTER(CONTAINS(STR(?type), "GHGEmissionsMetric"))
    ?metric rdfs:label ?label .
    OPTIONAL { ?metric ont:hasValue ?value }
    OPTIONAL { ?metric ont:hasUnit ?unit }
    OPTIONAL { ?metric ont:hasYear ?year }
}
```

The `explore_rdf_entities()` helper function in `queries.py` executes these SPARQL queries on the RDF graph and returns a structured summary. See the code cell below for usage.

In [6]:
# Inspect extracted entities from EcoSteel report using RDF
from queries import explore_rdf_entities
import glob

ttl_files = glob.glob("output/**/ecosteel*.ttl", recursive=True)

if ttl_files:
    print(f"📄 Parsing RDF file: {ttl_files[0]}\n")

    summary = explore_rdf_entities(ttl_files[0])

    print("=" * 70)
    print("EXTRACTED ENTITIES SUMMARY (Key Types Only)")
    print("=" * 70)
    print("\nKey climate reporting entities:")
    for type_name, count in summary["type_counts"].items():
        print(f"  • {type_name}: {count}")

    print(
        f"\n({sum(summary['type_counts'].values())} key entities shown, {summary['total_entities']} total entities extracted)"
    )

    print("\n" + "=" * 70)
    print("SAMPLE: GHG EMISSIONS METRICS (Scope 1, 2, 3)")
    print("=" * 70)

    for sample in summary["ghg_samples"]:
        print(
            f"  • {sample['label']}: {sample['value']} {sample['unit']} ({sample['year']})"
        )

    print("\n" + "=" * 70)
    print(f"✓ Total triples in graph: {summary['total_triples']}")
    print("✓ The ontology guided Perseus to extract structured, queryable entities")
else:
    print("⚠️  No .ttl file found yet. Run previous cells first.")


📄 Parsing RDF file: output/ecosteel_climate_graph.ttl

EXTRACTED ENTITIES SUMMARY (Key Types Only)

Key climate reporting entities:
  • GHGEmissionsMetric: 8
  • ReductionTarget: 4
  • StrategicPillar: 3
  • CarbonIntensityMetric: 2
  • Commitment: 1
  • TransitionRisk: 1
  • StrategicProgram: 1
  • PhysicalRisk: 1

(21 key entities shown, 52 total entities extracted)

SAMPLE: GHG EMISSIONS METRICS (Scope 1, 2, 3)
  • Scope 1 Emissions 2024: 1200000 tCO2e (2024)
  • Scope 2 Emissions 2024: 180000 tCO2e (2024)
  • Scope 3 Emissions 2024: 2400000 tCO2e (2024)

✓ Total triples in graph: 248
✓ The ontology guided Perseus to extract structured, queryable entities


### What This Means

Perseus extracted **8 GHGEmissionsMetric entities** - covering Scope 1, 2, 3 and their sub-categories (transport, raw materials, product use). This comprehensive extraction means we can now:
- Query emissions by scope: *"Show all Scope 3 upstream emissions"*
- Aggregate totals: *"What's the total carbon footprint?"*
- Compare categories: *"Which Scope 3 category dominates?"*

The ontology also captured **3 StrategicPillar entities** linked to the GreenSteel 2030 program, preserving the narrative structure that explains *how* EcoSteel plans to reduce these emissions.

Without ontology guidance, an LLM might extract these metrics as unstructured text or with inconsistent units (some in kg, some in tons). The ontology ensures all emissions are normalized to `tCO2e` and properly typed as `GHGEmissionsMetric` with `hasScope`, `hasYear`, and `hasUnit` properties.

## See the Transformation: Unstructured Text → CSRD-Compliant Entities

To better illustrate how Perseus transforms unstructured text into structured knowledge, here are concrete examples showing **the complete pattern: Ontology schema → Source text → Extracted RDF entities**.
This demonstrates how the ontology guides the LLM to extract and structure information:

1. **Quantitative disclosure** (structured numbers): GHG emissions metric (from EcoSteel report)
2. **Narrative strategic element** (contextual information): Strategic program with pillars (from EcoSteel report)
3. **Narrative risk element** (contextual information): Climate risk description (from EcoSteel report)
4. **Forward-looking commitment** (target with validation): Net-Zero commitment (from TechGreen report)

---

### Example 1: Quantitative Disclosure (Structured Numbers) - from EcoSteel report

**🔷 ONTOLOGY CLASSES & RELATIONS USED:**

| Class | Description |
|-------|-------------|
| `GHGEmissionsMetric` | Greenhouse gas emissions measurement |
| `Company` | Organization entity |

| Relation | Description |
|----------|-------------|
| `hasMetric` | Links company to its metrics |

| Property | Description |
|----------|-------------|
| `hasValue` | Numeric value of the metric |
| `hasUnit` | Unit of measurement (tCO2e) |
| `hasYear` | Reporting year |
| `hasScope` | GHG Protocol scope (1, 2, or 3) |

**📄 SOURCE TEXT (*annual_report.md*):**
```
"Scope 1: 1,200,000 tCO2e (direct combustion, blast furnaces)"
```

**🔍 EXTRACTED ENTITY:**
```yaml
Type: GHGEmissionsMetric
Label: Scope 1 Emissions 2024
Value: 1,200,000
Unit: tCO2e
Year: 2024
```

---

### Example 2: Strategic Program with Pillar - from EcoSteel report

**🔷 ONTOLOGY CLASSES & RELATIONS USED:**

| Class | Description |
|-------|-------------|
| `StrategicProgram` | Company-wide strategic initiative |
| `StrategicPillar` | Major component of a strategic program |

| Relation | Description |
|----------|-------------|
| `hasPillar` | Links program to its pillars |

| Property | Description |
|----------|-------------|
| `label` | Name/title of the entity |
| `hasBudget` | Financial budget allocated |

**📄 SOURCE TEXT (*annual_report.md*):**
```
"GreenSteel 2030 Program - Cross-functional strategic program with €850M budget."
"Pillar 1: Process Electrification - Progressive conversion of blast furnaces to DRI-EAF technology."
```

**🔍 EXTRACTED ENTITIES:**

**Entity 1 (Program):**
```yaml
Type: StrategicProgram
Label: "GreenSteel 2030" Program
Budget: €850M
Description: Cross-functional strategic program
```

**Entity 2 (Pillar):**
```yaml
Type: StrategicPillar
Label: Process Electrification
Description: Progressive conversion of blast furnaces to DRI-EAF
```

**Relationship:**
```
(GreenSteel 2030)-[:hasPillar]->(Process Electrification)
```

---

### Example 3: Climate Risk - from EcoSteel report

**🔷 ONTOLOGY CLASSES & RELATIONS USED:**

| Class | Description |
|-------|-------------|
| `PhysicalRisk` | Climate-related physical risk (TCFD category) |

| Relation | Description |
|----------|-------------|
| `hasRisk` | Links company to identified risks |

**📄 SOURCE TEXT (*annual_report.md*):**
```
"Physical risks: extreme weather events (floods, storms) affecting logistics and supply chains"
```

**🔍 EXTRACTED ENTITY:**
```yaml
Type: PhysicalRisk
Label: Extreme Weather Events Risk
```

**Relationship:**
```
(EcoSteel Industries)-[:hasRisk]->(Extreme Weather Events Risk)
```

---

### Example 4: Net-Zero Commitment - from TechGreen report

**🔷 ONTOLOGY CLASSES & RELATIONS USED:**

| Class | Description |
|-------|-------------|
| `Commitment` | Long-term climate commitment |
| `ReductionTarget` | Quantified emission reduction goal |
| `Organization` | External validation body (e.g., SBTi) |

| Relation | Description |
|----------|-------------|
| `hasCommitment` | Links company to its commitments |
| `hasTarget` | Links company to specific targets |
| `isSubmittedTo` | Submission to external validation body |

| Property | Description |
|----------|-------------|
| `hasTargetYear` | Year when target should be achieved |
| `hasBaselineYear` | Reference year for reduction calculations |
| `hasReductionValue` | Percentage or absolute reduction |

**📄 SOURCE TEXT (press_release.md*):**
```
"Net-Zero 2040 Commitment - 10 years ahead of Paris Agreement target
Application submitted to Science Based Targets initiative in December 2024
2030 intermediate target: -55% reduction in Scope 1+2+3 emissions (2024 baseline)"
```

**🔍 EXTRACTED ENTITIES:**

**Entity 1 (Commitment):**
```yaml
Type: Commitment
Label: Net-Zero 2040 commitment
```

**Entity 2 (Reduction Target):**
```yaml
Type: ReductionTarget
Label: -55% reduction target by 2030
ReductionValue: 55%
TargetYear: 2030
BaselineYear: 2024
```

**Entity 3 (Validation Body):**
```yaml
Type: Organization
Label: Science Based Targets initiative
```

**Relationships:**
```
(TechGreen Solutions)-[:hasCommitment]->(Net-Zero 2040 commitment)
(TechGreen Solutions)-[:hasTarget]->(-55% reduction target by 2030)
(Net-Zero 2040 commitment)-[:isSubmittedTo]->(Science Based Targets initiative)
```

---

### 💡 Key Insight

The ontology provides the **schema** that guides the LLM to:
- **Recognize** which information is relevant (e.g., distinguishing a "target" from a "metric")
- **Structure** it correctly (e.g., linking pillars to programs, targets to commitments)
- **Normalize** values (e.g., years as xsd:gYear, numbers with units)

Without the ontology, LLMs produce unstructured JSON. With it, you get **queryable, CSRD-compliant knowledge graphs**.

## 6. ESRS E1 Compliance Verification

We run **automated compliance checks** on our Neo4j graph to verify 5 ESRS E1 indicators for both companies.

**The code below:**
1. Connects to Neo4j and runs Cypher queries for each indicator
2. Checks 3 quantitative disclosures: GHG emissions (E1-6), carbon intensity (E1-5), Net-Zero targets (E1-4)
3. Checks 2 narrative disclosures: strategic programs with pillars, climate risks
4. Outputs a compliance scorecard per company

This demonstrates how structured graphs enable automated regulatory verification at scale.

**Note:** This time we are using **Cypher** to query the **local Neo4j graph database**, optimized for complex graph traversals and production queries across multiple companies.

### Cypher Query Examples

The compliance checks use Cypher queries defined in `queries.py`. Here two examples key patterns for verifying ESRS E1 indicators:

**Cypher - Check GHG emissions disclosure (E1-6):**
```cypher
MATCH (c:Company {label: $name})-[:hasMetric]->(m:GHGEmissionsMetric)
RETURN count(m) > 0 AS present
```

**Cypher - Check strategic program with pillars:**
```cypher
MATCH (c:Company {label: $name})-[:hasProgram]->(p:StrategicProgram)-[:hasPillar]->(pillar)
RETURN count(pillar) > 0 AS present
```


The `check_esrs_e1_compliance()` helper function in `queries.py` executes 5 Cypher queries for a given company and returns a compliance report. The `print_compliance_report()` function formats the results. See the code cell below for usage.

In [7]:
# CSRD ESRS E1 Compliance Check - Quantitative AND Narrative
from queries import check_esrs_e1_compliance, print_compliance_report
from neo4j import GraphDatabase
import os

# Connect to Neo4j
uri = os.getenv("NEO4J_URI", "bolt://localhost:7687")
user = os.getenv("NEO4J_USER", "neo4j")
password = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(uri, auth=(user, password))

print("📋 CSRD ESRS E1 Compliance Verification")
print("=" * 70)

companies = ["EcoSteel Industries", "TechGreen Solutions"]

with driver.session() as session:
    for company in companies:
        results = session.execute_read(check_esrs_e1_compliance, company)
        print_compliance_report(results, company)

driver.close()
print("\n" + "=" * 70)


📋 CSRD ESRS E1 Compliance Verification

🏢 EcoSteel Industries
  Quantitative (structured numbers):
    ✓ E1-6: GHG emissions disclosed
    ✓ E1-5: Carbon intensity disclosed
    ✓ E1-4: Net-Zero target set
  Narrative (contextual information):
    ✓ Strategic program with pillars
    ✓ Climate risks with description

  5/5 indicators (100%)

🏢 TechGreen Solutions
  Quantitative (structured numbers):
    ✓ E1-6: GHG emissions disclosed
    ✓ E1-5: Carbon intensity disclosed
    ✓ E1-4: Net-Zero target set
  Narrative (contextual information):
    ✓ Strategic program with pillars
    ✓ Climate risks with description

  5/5 indicators (100%)



### Compliance Results

The verification shows that **both companies meet all 5 ESRS E1 indicators** we checked:

**Quantitative KPIs (structured numbers):**
- ✓ **E1-6**: Both companies disclosed GHG emissions by scope (Scope 1, 2, 3)
- ✓ **E1-5**: Both reported carbon intensity metrics (emissions per unit of production/activity)
- ✓ **E1-4**: Both set Net-Zero targets with clear timelines (EcoSteel 2050, TechGreen 2040)

**Narrative context (strategic and risk information):**
- ✓ Both companies described structured climate action programs with strategic pillars (GreenSteel 2030, Digital4Climate)
- ✓ Both identified climate risks (physical: extreme weather/water stress; transition: technological obsolescence/carbon pricing)

This demonstrates that Perseus successfully extracted and structured all the key climate data required by ESRS E1, enabling automated verification that would otherwise require manual review of lengthy reports.

## 7. Conclusion

We processed 2 unstructured climate reports into a unified knowledge graph and verified 5 ESRS E1 indicators:
- **3 quantitative** (GHG emissions, carbon intensity, Net-Zero targets)
- **2 narrative** (strategic programs with pillars, climate risks with descriptions)

#### Why This Matters

Traditional approaches to ESG data extraction face three challenges:
1. **Manual extraction** - time-consuming, error-prone, doesn't scale
2. **Generic LLMs** - produce unstructured JSON, inconsistent units, no semantic relationships
3. **XBRL tagging** - captures numbers but misses narrative context (strategy descriptions, risk assessments)

**Perseus + Ontology** bridges this gap: structured extraction that scales, with semantic normalization and preserved narrative context.

#### Real-World Applications

Once reports become queryable graphs, these analyses shift from weeks of manual work to minutes of query execution.

**ESG Analysts**  
Compare climate strategies across sectors at scale:
- *"Show all green hydrogen programs with their budgets and timelines"*
- *"Compare physical vs transition risk assessments in the steel industry"*

**Regulators & Standard Setters**  
Detect disclosure gaps and benchmark compliance:
- *"List companies missing physical climate risk assessments"*
- *"Analyze adoption rates of SBTi-validated targets by sector"*

---

#### Next Steps

- **Custom ontologies**: Adapt the CSRD ontology to your specific regulatory framework or use Lettria's Ontology Toolkit to build domain-specific ontologies.
- **Explore more**: Visit [Lettria resources](https://www.lettria.com/lettria-lab) for additional ontologies and documentation.

Questions? Contact the Lettria team or explore the [Perseus documentation](https://github.com/Lettria/perseus-client).